In [1]:
import numpy as np
import pandas as pd

In [2]:
contestKey = '161756118'
# lineup_df = pd.read_csv(rf"C:\Users\James\Documents\MLB\Data\C02. Optimization\2. Lineups\Lineups {contestKey}.csv")
# field_lineup_df = pd.read_csv(rf"C:\Users\James\Documents\MLB\Data\C02. Optimization\2B. Field Lineups\Lineups {contestKey}.csv")
# player_df = pd.read_csv(rf"C:\Users\James\Documents\MLB\Data\C02. Optimization\1. Players\Players {contestKey}.csv")
# payout_df = pd.read_csv(rf"C:\Users\James\Documents\MLB\Data\A09. DraftKings\3. Payouts\Payouts {contestKey}.csv")
# contest_df = pd.read_csv(rf"C:\Users\James\Documents\MLB\Data\B03. Contest Guides\Contest Guide {contestKey}.csv")

In [ ]:
contest_df.head()

In [ ]:
player_df['Name + ID'] = player_df['Name + ID'].str.replace(" (", "(", regex=False)

In [ ]:
entry_fee = 4
portfolio_size = 1
n_iterations=200
swap_size=2
optimize_metric="EV"  # 'EV', 'Top1pct_rate', 'Top5pct_rate'

contest_size = contest_df['entries'].iloc[0]
contest_size

In [ ]:
import numpy as np
import pandas as pd

# # ===============================
# # 0. Prepare unified identifier
# # ===============================
# player_df_indexed = player_df.set_index('Name + ID')

# # ===============================
# # 1. Define player slot columns
# # ===============================
# player_cols = ['P','P.1','C','1B','2B','3B','SS','OF','OF.1','OF.2']

# ===============================
# 2. Build payout array
# ===============================
def build_payout_array(payout_df, contest_size):
    payout_array = np.zeros(contest_size)
    for _, row in payout_df.iterrows():
        start = int(row['minPosition']) - 1
        end = int(row['maxPosition'])
        payout = float(str(row['payoutDescription']).replace('$','').replace(',', ''))
        payout_array[start:end] = payout
    if len(payout_array) < contest_size:
        payout_array = np.pad(payout_array, (0, contest_size - len(payout_array)))
    return payout_array

# ===============================
# 3. Build score matrices
# ===============================
def build_score_matrix(lineup_df, player_df_indexed, player_cols):
    sim_cols = [c for c in player_df_indexed.columns if c.startswith('FP_') or c.startswith('sim_')]
    score_matrix = []

    for _, lineup in lineup_df.iterrows():
        player_ids = [lineup[col] for col in player_cols]
        sims = player_df_indexed.loc[player_ids, sim_cols].sum(axis=0).values
        score_matrix.append(sims)

    return np.array(score_matrix)

# ===============================
# 4. Evaluate portfolio
# ===============================
def evaluate_portfolio(selected_idx, my_scores, field_scores, payout_array, entry_fee, contest_size):
    n_my = len(selected_idx)
    n_field = field_scores.shape[0]
    n_sims = my_scores.shape[1]

    # Compute how many field lineups we can realistically add
    n_field_needed = min(contest_size - n_my, n_field)
    n_total_simulated = n_my + n_field_needed

    per_lineup_payouts = np.zeros((n_my, n_sims))
    per_lineup_top1 = np.zeros((n_my, n_sims))
    per_lineup_top5 = np.zeros((n_my, n_sims))

    for s in range(n_sims):
        # Sample field lineups if needed
        if n_field_needed > 0:
            sampled_field = np.random.choice(field_scores[:, s], size=n_field_needed, replace=False)
            all_scores = np.concatenate([my_scores[selected_idx, s], sampled_field])
        else:
            all_scores = my_scores[selected_idx, s]

        ranks = np.argsort(-all_scores).argsort()  # 0 = best
        portfolio_ranks = ranks[:n_my]

        # Use actual number of competitors in simulation to set thresholds
        n_competitors = len(all_scores)
        top1_threshold = max(1, int(np.ceil(0.01 * n_competitors)))
        top5_threshold = max(1, int(np.ceil(0.05 * n_competitors)))

        rank_clipped = np.minimum(portfolio_ranks, len(payout_array) - 1)
        per_lineup_payouts[:, s] = payout_array[rank_clipped]

        per_lineup_top1[:, s] = (portfolio_ranks < top1_threshold).astype(float)
        per_lineup_top5[:, s] = (portfolio_ranks < top5_threshold).astype(float)

    per_lineup_ev = per_lineup_payouts.mean(axis=1)
    per_lineup_top1_rate = per_lineup_top1.mean(axis=1)
    per_lineup_top5_rate = per_lineup_top5.mean(axis=1)

    portfolio_ev = per_lineup_ev.sum() - entry_fee * n_my
    portfolio_roi = portfolio_ev / (entry_fee * n_my) if entry_fee > 0 else 0
    portfolio_std = per_lineup_payouts.sum(axis=0).std()
    portfolio_top1_rate = per_lineup_top1_rate.mean()
    portfolio_top5_rate = per_lineup_top5_rate.mean()

    portfolio_metrics = {
        "EV": portfolio_ev,
        "ROI": portfolio_roi,
        "StdDev": portfolio_std,
        "Top_1pct_rate": portfolio_top1_rate,
        "Top_5pct_rate": portfolio_top5_rate
    }

    return portfolio_metrics, per_lineup_ev, per_lineup_top1_rate, per_lineup_top5_rate

# ===============================
# 5. Monte Carlo portfolio search
# ===============================
def monte_carlo_portfolio_search(lineup_df, my_scores, field_scores, payout_array, entry_fee,
                                 contest_size, portfolio_size=20, n_iterations=1000, swap_size=2,
                                 random_seed=42, optimize_metric="Top_1pct_rate"):

    np.random.seed(random_seed)
    n_candidates = len(lineup_df)
    n_field = field_scores.shape[0]

    if n_field + portfolio_size < contest_size:
        print(f"WARNING: Simulated field + portfolio ({n_field + portfolio_size}) < contest size ({contest_size}). "
            "EV may be overestimated.")

    projected_fp = lineup_df['FPPG'].values
    top_indices = np.argsort(-projected_fp)[:portfolio_size]
    best_portfolio = top_indices.copy()

    # initial evaluation (no warning needed here)
    metrics, _, _, _ = evaluate_portfolio(best_portfolio, my_scores, field_scores,
                                          payout_array, entry_fee, contest_size)
    best_value = metrics[optimize_metric]

    for it in range(n_iterations):
        swap_out = np.random.choice(best_portfolio, size=min(swap_size, portfolio_size), replace=False)
        remaining_candidates = np.setdiff1d(np.arange(n_candidates), best_portfolio)
        if len(remaining_candidates) == 0:
            continue
        swap_in = np.random.choice(remaining_candidates, size=len(swap_out), replace=False)
        new_portfolio = best_portfolio.copy()
        for o, i in zip(swap_out, swap_in):
            new_portfolio[np.where(new_portfolio == o)[0][0]] = i

        metrics, _, _, _ = evaluate_portfolio(new_portfolio, my_scores, field_scores,
                                              payout_array, entry_fee, contest_size)
        new_value = metrics[optimize_metric]
        if new_value > best_value:
            best_portfolio = new_portfolio
            best_value = new_value
            print(f"Iteration {it+1}: New best {optimize_metric} = {best_value:.4f}")

    portfolio_metrics, per_lineup_ev, per_lineup_top1_rate, per_lineup_top5_rate = evaluate_portfolio(
        best_portfolio, my_scores, field_scores, payout_array, entry_fee, contest_size
    )

    selected_lineups_df = lineup_df.iloc[best_portfolio].copy()
    selected_lineups_df["EV_Payout"] = per_lineup_ev
    selected_lineups_df["Top1pct_rate"] = per_lineup_top1_rate
    selected_lineups_df["Top5pct_rate"] = per_lineup_top5_rate

    return best_portfolio, portfolio_metrics, selected_lineups_df

# # ===============================
# # 6. Example usage
# # ===============================
# payout_array = build_payout_array(payout_df, contest_size)
# my_scores = build_score_matrix(lineup_df, player_df_indexed, player_cols)
# field_scores = build_score_matrix(field_lineup_df, player_df_indexed, player_cols)

# best_idx, portfolio_metrics, selected_lineups_df = monte_carlo_portfolio_search(
#     lineup_df, my_scores, field_scores, payout_array, entry_fee=entry_fee,
#     contest_size=contest_size, portfolio_size=portfolio_size, n_iterations=n_iterations,
#     swap_size=swap_size, optimize_metric=optimize_metric
# )

In [ ]:
def choose_portfolio(contestKey, portfolio_size=20, n_iterations=1000, swap_size=2, random_seed=42, optimize_metric="Top_1pct_rate"):
    # Payout array
    def build_payout_array(payout_df, contest_size):
        payout_array = np.zeros(contest_size)
        for _, row in payout_df.iterrows():
            start = int(row['minPosition']) - 1
            end = int(row['maxPosition'])
            payout = float(str(row['payoutDescription']).replace('$','').replace(',', ''))
            payout_array[start:end] = payout
        if len(payout_array) < contest_size:
            payout_array = np.pad(payout_array, (0, contest_size - len(payout_array)))
        return payout_array

    # Score matrix
    def build_score_matrix(lineup_df, player_df_indexed, player_cols):
        sim_cols = [c for c in player_df_indexed.columns if c.startswith('FP_') or c.startswith('sim_')]
        score_matrix = []

        for _, lineup in lineup_df.iterrows():
            player_ids = [lineup[col] for col in player_cols]
            sims = player_df_indexed.loc[player_ids, sim_cols].sum(axis=0).values
            score_matrix.append(sims)

        return np.array(score_matrix)

    # Portfolio evaluation
    def evaluate_portfolio(selected_idx, my_scores, field_scores, payout_array, entry_fee, contest_size):
        n_my = len(selected_idx)
        n_field = field_scores.shape[0]
        n_sims = my_scores.shape[1]

        # Compute how many field lineups we can realistically add
        n_field_needed = min(contest_size - n_my, n_field)
        # n_total_simulated = n_my + n_field_needed

        per_lineup_payouts = np.zeros((n_my, n_sims))
        per_lineup_top1 = np.zeros((n_my, n_sims))
        per_lineup_top5 = np.zeros((n_my, n_sims))

        for s in range(n_sims):
            # Sample field lineups if needed
            if n_field_needed > 0:
                sampled_field = np.random.choice(field_scores[:, s], size=n_field_needed, replace=False)
                all_scores = np.concatenate([my_scores[selected_idx, s], sampled_field])
            else:
                all_scores = my_scores[selected_idx, s]

            ranks = np.argsort(-all_scores).argsort()  # 0 = best
            portfolio_ranks = ranks[:n_my]

            # Use actual number of competitors in simulation to set thresholds
            n_competitors = len(all_scores)
            top1_threshold = max(1, int(np.ceil(0.01 * n_competitors)))
            top5_threshold = max(1, int(np.ceil(0.05 * n_competitors)))

            rank_clipped = np.minimum(portfolio_ranks, len(payout_array) - 1)
            per_lineup_payouts[:, s] = payout_array[rank_clipped]

            per_lineup_top1[:, s] = (portfolio_ranks < top1_threshold).astype(float)
            per_lineup_top5[:, s] = (portfolio_ranks < top5_threshold).astype(float)

        per_lineup_ev = per_lineup_payouts.mean(axis=1)
        per_lineup_top1_rate = per_lineup_top1.mean(axis=1)
        per_lineup_top5_rate = per_lineup_top5.mean(axis=1)

        portfolio_ev = per_lineup_ev.sum() - entry_fee * n_my
        portfolio_roi = portfolio_ev / (entry_fee * n_my) if entry_fee > 0 else 0
        portfolio_std = per_lineup_payouts.sum(axis=0).std()
        portfolio_top1_rate = per_lineup_top1_rate.mean()
        portfolio_top5_rate = per_lineup_top5_rate.mean()

        portfolio_metrics = {
            "EV": portfolio_ev,
            "ROI": portfolio_roi,
            "StdDev": portfolio_std,
            "Top_1pct_rate": portfolio_top1_rate,
            "Top_5pct_rate": portfolio_top5_rate
        }

        return portfolio_metrics, per_lineup_ev, per_lineup_top1_rate, per_lineup_top5_rate

    # Monte Carlo portfolio search
    def monte_carlo_portfolio_search(lineup_df, my_scores, field_scores, payout_array, entry_fee,
                                    contest_size, portfolio_size=20, n_iterations=1000, swap_size=2,
                                    random_seed=42, optimize_metric="Top_1pct_rate"):

        np.random.seed(random_seed)
        n_candidates = len(lineup_df)
        n_field = field_scores.shape[0]

        if n_field + portfolio_size < contest_size:
            print(f"WARNING: Simulated field + portfolio ({n_field + portfolio_size}) < contest size ({contest_size}). "
                "EV may be overestimated.")

        projected_fp = lineup_df['FPPG'].values
        top_indices = np.argsort(-projected_fp)[:portfolio_size]
        best_portfolio = top_indices.copy()

        # initial evaluation (no warning needed here)
        metrics, _, _, _ = evaluate_portfolio(best_portfolio, my_scores, field_scores,
                                            payout_array, entry_fee, contest_size)
        best_value = metrics[optimize_metric]

        for it in range(n_iterations):
            swap_out = np.random.choice(best_portfolio, size=min(swap_size, portfolio_size), replace=False)
            remaining_candidates = np.setdiff1d(np.arange(n_candidates), best_portfolio)
            if len(remaining_candidates) == 0:
                continue
            swap_in = np.random.choice(remaining_candidates, size=len(swap_out), replace=False)
            new_portfolio = best_portfolio.copy()
            for o, i in zip(swap_out, swap_in):
                new_portfolio[np.where(new_portfolio == o)[0][0]] = i

            metrics, _, _, _ = evaluate_portfolio(new_portfolio, my_scores, field_scores,
                                                payout_array, entry_fee, contest_size)
            new_value = metrics[optimize_metric]
            if new_value > best_value:
                best_portfolio = new_portfolio
                best_value = new_value
                print(f"Iteration {it+1}: New best {optimize_metric} = {best_value:.4f}")

        portfolio_metrics, per_lineup_ev, per_lineup_top1_rate, per_lineup_top5_rate = evaluate_portfolio(
            best_portfolio, my_scores, field_scores, payout_array, entry_fee, contest_size
        )

        selected_lineups_df = lineup_df.iloc[best_portfolio].copy()
        selected_lineups_df["EV_Payout"] = per_lineup_ev
        selected_lineups_df["Top1pct_rate"] = per_lineup_top1_rate
        selected_lineups_df["Top5pct_rate"] = per_lineup_top5_rate

        return best_portfolio, portfolio_metrics, selected_lineups_df



    # Read in necessary datasets
    try:
        lineup_df = pd.read_csv(rf"C:\Users\James\Documents\MLB\Data\C02. Optimization\2. Lineups\Lineups {contestKey}.csv")
        field_lineup_df = pd.read_csv(rf"C:\Users\James\Documents\MLB\Data\C02. Optimization\2B. Field Lineups\Lineups {contestKey}.csv")
        player_df = pd.read_csv(rf"C:\Users\James\Documents\MLB\Data\C02. Optimization\1. Players\Players {contestKey}.csv")
        payout_df = pd.read_csv(rf"C:\Users\James\Documents\MLB\Data\A09. DraftKings\3. Payouts\Payouts {contestKey}.csv")
        contest_df = pd.read_csv(rf"C:\Users\James\Documents\MLB\Data\B03. Contest Guides\Contest Guide {contestKey}.csv")
    except Exception as e:
        print(f"Error reading files for contest {contestKey}: {e}")
        return None, None, None


    # Clean player file Name + IDs to match lineup format
    player_df['Name + ID'] = player_df['Name + ID'].str.replace(" (", "(", regex=False)

    # Index player Name + ID for quick lookup
    player_df_indexed = player_df.set_index('Name + ID')

    # Set player position columns
    player_cols = ['P','P.1','C','1B','2B','3B','SS','OF','OF.1','OF.2']

    # Gather information from contest guide
    contest_size = contest_df['entries'].iloc[0]
    entry_fee = contest_df['entryFee'].iloc[0]

    # Create payout and score arrays
    payout_array = build_payout_array(payout_df, contest_size)
    my_scores = build_score_matrix(lineup_df, player_df_indexed, player_cols)
    field_scores = build_score_matrix(field_lineup_df, player_df_indexed, player_cols)

    # Run portfolio optimization
    best_idx, portfolio_metrics, selected_lineups_df = monte_carlo_portfolio_search(lineup_df, my_scores, field_scores, payout_array, entry_fee, contest_size, portfolio_size, n_iterations, swap_size, random_seed, optimize_metric)

    print(f"Finished running contest {contestKey} portfolio optimization. Best {optimize_metric}: {portfolio_metrics[optimize_metric]:.4f}")


    return best_idx, portfolio_metrics, selected_lineups_df

In [4]:
best_idx, portfolio_metrics, selected_lineups_df = choose_portfolio(contestKey, portfolio_size=20, n_iterations=100, swap_size=2, random_seed=42, optimize_metric="EV")



Iteration 1: New best EV = 118.4531
Iteration 3: New best EV = 121.0898
Iteration 6: New best EV = 123.3818
Iteration 8: New best EV = 130.0361
Iteration 10: New best EV = 131.9658
Iteration 12: New best EV = 133.9150
Iteration 13: New best EV = 135.2178
Iteration 14: New best EV = 136.7275
Iteration 29: New best EV = 138.1064
Iteration 35: New best EV = 139.9795
Iteration 40: New best EV = 142.9912
Iteration 55: New best EV = 144.5791
Iteration 60: New best EV = 145.3809
Iteration 62: New best EV = 146.4561
Iteration 69: New best EV = 146.8867
Iteration 73: New best EV = 148.9736
Iteration 76: New best EV = 150.1230
Iteration 81: New best EV = 152.0625
Finished running contest 161756118 portfolio optimization. Best EV: 149.5488


In [5]:
selected_lineups_df.head()

,P,P.1,C,1B,2B,3B,SS,OF,OF.1,OF.2,Budget,FPPG,stack,EV_Payout,Top1pct_rate,Top5pct_rate
747,Hunter Greene(34262940),Landon Knack(34263170),Carson Kelly(34263014),Spencer Torkelson(34262967),Thairo Estrada(34262977),Matt Chapman(34262971),Javier Baez(34263006),Matt Vierling(34262973),Shohei Ohtani(34262943),Wenceel Perez(34262959),49200.0,99.352,"[5, 2, 1]",10.758789,0.052734,0.167969
546,Hunter Greene(34262940),Landon Knack(34263170),Carson Kelly(34263014),LaMonte Wade Jr.(34262981),Thairo Estrada(34262977),Andy Ibanez(34262984),Elly De La Cruz(34262946),Mark Canha(34262955),Matt Vierling(34262973),Wenceel Perez(34262959),49800.0,99.891,"[5, 2, 1]",11.437500,0.077148,0.203125
688,Jordan Hicks(34262939),Hunter Greene(34262940),Carson Kelly(34263014),Christian Walker(34262954),Andy Ibanez(34262984),Gio Urshela(34263007),Elly De La Cruz(34262946),Joc Pederson(34262960),Mark Canha(34262955),Matt Vierling(34262973),49300.0,99.484,"[5, 2, 1]",15.047852,0.080078,0.192383
835,Jordan Hicks(34262939),Hunter Greene(34262940),Carson Kelly(34263014),Christian Walker(34262954),Ketel Marte(34262948),Gio Urshela(34263007),Kevin Newman(34263003),Corbin Carroll(34262956),Joc Pederson(34262960),Mark Canha(34262955),49300.0,98.537,"[5, 3]",8.416016,0.040039,0.143555
560,Hunter Greene(34262940),Landon Knack(34263170),Curt Casali(34263021),LaMonte Wade Jr.(34262981),Thairo Estrada(34262977),Matt Chapman(34262971),Mookie Betts(34262945),Mike Yastrzemski(34262988),Corbin Carroll(34262956),Joc Pederson(34262960),49600.0,99.854,"[5, 2, 1]",7.685547,0.032227,0.116211
